In [1]:
import os

import pandas as pd
import numpy as np

import functions.cnn_helpers as help_funcs

# DNA MERFISH data (Su et al. 2020)

## Read distance info and save as arrays

In [2]:
dist_data = pd.read_csv('../data/Su_2020/selected_structures_1000_nm_thr.tsv', sep='\t')
transc_data = pd.read_csv('../data/Su_2020/transcriptional_data_1000_nm_thr.tsv', sep='\t')

dataset = pd.merge(dist_data, transc_data.iloc[:, :2], how='left', on='Chromosome copy number')

n_loci = 38

### Iterate through all structures and save them as an image

In [3]:
for id, row in dataset.iterrows():
    
    struct_id = int(row['Chromosome copy number'])
    state = int(row['BACH1'])
    if state == 0:
        state_save = 'inactive'
    else:
        state_save = 'active'
    
    dists = row.iloc[1 : -1]
    
    locus_i = 0
    dist_i = np.zeros((n_loci, n_loci))
    
    for i in range(0, n_loci):
        for j in range(i + 1, n_loci):
            dist_i[i,j] = dists.iloc[locus_i]
            
            locus_i += 1
    
    dist_i = dist_i + dist_i.T
    
    
    dist_i = pd.DataFrame(dist_i)
    dist_i.to_csv(f"../data/Su_2020/distance_maps/{state_save}/structure_{struct_id}.tsv", sep='\t', header = False, index=False)

## Create splits and save them before (.png) and after (.tsv) normalization

In [ ]:
help_funcs.kfold_splits(src_folder = '../data/Su_2020/distance_maps/', dst_folder = '../data/Su_2020/CNN_splits')

FileNotFoundError: [Errno 2] No such file or directory: '../data/distance_maps/'

In [29]:
samples = []

# Subfolders are class labels
classes = sorted(
    [
        class_i
        for class_i in os.listdir('../data/CNN_splits_zscore/fold_0/test')
        if os.path.isdir(os.path.join('../data/CNN_splits_zscore/fold_0/test', class_i))
    ],
    reverse=True,
)
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

for label in os.listdir('../data/CNN_splits_zscore/fold_0/test'):
    class_dir = os.path.join('../data/CNN_splits_zscore/fold_0/test', label)
    if not os.path.isdir(class_dir):
        continue

    for fname in os.listdir(class_dir):
        if fname.endswith(".tsv"):
            path = os.path.join(class_dir, fname)
            samples.append((path, int(class_to_idx[label])))


In [32]:
samples

[('../data/CNN_splits_zscore/fold_0/test/inactive/structure_1413.tsv', 0),
 ('../data/CNN_splits_zscore/fold_0/test/inactive/structure_2126.tsv', 0),
 ('../data/CNN_splits_zscore/fold_0/test/inactive/structure_6432.tsv', 0),
 ('../data/CNN_splits_zscore/fold_0/test/inactive/structure_8379.tsv', 0),
 ('../data/CNN_splits_zscore/fold_0/test/inactive/structure_1177.tsv', 0),
 ('../data/CNN_splits_zscore/fold_0/test/inactive/structure_3006.tsv', 0),
 ('../data/CNN_splits_zscore/fold_0/test/inactive/structure_9271.tsv', 0),
 ('../data/CNN_splits_zscore/fold_0/test/inactive/structure_1836.tsv', 0),
 ('../data/CNN_splits_zscore/fold_0/test/inactive/structure_10542.tsv', 0),
 ('../data/CNN_splits_zscore/fold_0/test/inactive/structure_8755.tsv', 0),
 ('../data/CNN_splits_zscore/fold_0/test/inactive/structure_11476.tsv', 0),
 ('../data/CNN_splits_zscore/fold_0/test/inactive/structure_2091.tsv', 0),
 ('../data/CNN_splits_zscore/fold_0/test/inactive/structure_11675.tsv', 0),
 ('../data/CNN_splits_

# ORCA data (Mateo et al. 2020)

## Read distance info and save as arrays

In [59]:
dist_data = pd.read_csv('../data/Mateo_2019/selected_structures.tsv', sep='\t')
transc_data = pd.read_csv('../data/Mateo_2019/transcriptional_data.tsv', sep='\t')

dataset = pd.merge(dist_data, transc_data.iloc[:, :2], how='left', on='cellNumber')

n_loci = 52

In [60]:
dataset

,cellNumber,1-2,1-3,1-4,1-5,1-6,1-7,1-8,1-9,1-10,...,48-50,48-51,48-52,49-50,49-51,49-52,50-51,50-52,51-52,Abd-A_Intron
0,2.0,244.073929,156.705215,227.523438,298.594910,284.360779,319.927277,374.712128,263.243683,307.865479,...,158.210205,205.904953,89.207565,190.698318,243.541779,129.752823,129.892990,114.832573,134.459518,0
1,3.0,262.981628,149.599136,223.714142,298.312103,284.214844,319.122009,373.181580,263.169769,307.152435,...,148.777985,164.663910,56.323582,173.666397,188.108734,68.595428,74.033485,109.178902,135.220947,0
2,13.0,124.413574,62.232868,257.654388,220.720657,246.008240,441.668243,756.308289,263.075958,348.715790,...,359.092010,261.710297,175.725861,371.491577,260.087463,185.764450,168.830246,186.105453,112.900101,0
3,19.0,69.282951,138.565903,207.848846,443.208252,238.614304,273.902191,251.590515,290.168945,330.703491,...,247.873260,371.809601,341.640991,123.936684,247.873001,247.677017,123.936325,191.545700,206.775375,0
4,20.0,140.162399,280.324799,189.418167,158.130585,214.382599,314.234467,237.620895,364.475861,219.858322,...,26.428993,203.600006,384.076782,189.821518,270.037781,418.468781,180.703522,361.407043,180.703522,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2289,54296.0,100.621590,128.352631,286.576843,248.898636,218.128952,296.315460,379.651459,443.508423,539.908630,...,162.532089,142.924255,298.095154,138.703461,180.236008,301.980347,264.441345,421.862579,191.273575,1
2290,54298.0,90.672394,92.193672,265.648987,275.079193,197.236557,120.854897,159.037033,203.947113,274.588165,...,250.285370,146.688385,377.458160,144.669647,342.024048,612.465637,284.677338,569.354797,284.677460,1
2291,54310.0,223.677322,447.354645,531.770569,423.878387,492.909790,394.430939,300.274506,216.164169,359.377045,...,171.912140,257.868195,343.824280,85.956055,171.912109,257.868195,85.956055,171.912140,85.956085,1
2292,54333.0,45.941437,91.882996,295.733276,1146.179077,871.597900,598.217834,329.046478,502.175598,409.138489,...,160.223740,155.931351,269.772308,62.877518,141.710861,293.308624,157.828156,315.656555,157.828400,1


In [62]:
transc_data

,cellNumber,Abd-A_Intron
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0
...,...,...
54344,54360,1
54345,54361,0
54346,54362,0
54347,54363,0


### Iterate through all structures and save them as an image

In [51]:
for id, row in dataset.iterrows():
    
    struct_id = int(row['cellNumber'])
    state = int(row['Abd-A_Intron'])
    if state == 0:
        state_save = 'inactive'
    else:
        state_save = 'active'
    
    dists = row.iloc[1 : -1]
    
    locus_i = 0
    dist_i = np.zeros((n_loci, n_loci))
    
    for i in range(0, n_loci):
        for j in range(i + 1, n_loci):
            dist_i[i,j] = dists.iloc[locus_i]
            
            locus_i += 1
    
    dist_i = dist_i + dist_i.T
    
    
    dist_i = pd.DataFrame(dist_i)
    dist_i.to_csv(f"../data/Mateo_2019/distance_maps/{state_save}/structure_{struct_id}.tsv", sep='\t', header = False, index=False)

## Create splits and save them before (.png) and after (.tsv) normalization

In [56]:
help_funcs.kfold_splits(src_folder = '../data/Mateo_2019/distance_maps/', dst_folder = '../data/Mateo_2019/CNN_splits')

Creating fold 1/5...
Train size: 1376, Validation size: 459, Test size: 459
Creating fold 2/5...
Train size: 1376, Validation size: 459, Test size: 459
Creating fold 3/5...
Train size: 1376, Validation size: 459, Test size: 459
Creating fold 4/5...
Train size: 1376, Validation size: 459, Test size: 459
Creating fold 5/5...
Train size: 1376, Validation size: 459, Test size: 459


In [57]:
samples = []

# Subfolders are class labels
classes = sorted(
    [
        class_i
        for class_i in os.listdir('../data/Mateo_2019/CNN_splits_zscore/fold_0/test')
        if os.path.isdir(os.path.join('../data/Mateo_2019/CNN_splits_zscore/fold_0/test', class_i))
    ],
    reverse=True,
)
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

for label in os.listdir('../data/Mateo_2019/CNN_splits_zscore/fold_0/test'):
    class_dir = os.path.join('../data/Mateo_2019/CNN_splits_zscore/fold_0/test', label)
    if not os.path.isdir(class_dir):
        continue

    for fname in os.listdir(class_dir):
        if fname.endswith(".tsv"):
            path = os.path.join(class_dir, fname)
            samples.append((path, int(class_to_idx[label])))


In [58]:
samples

[('../data/Mateo_2019/CNN_splits_zscore/fold_0/test/inactive/structure_1361.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_zscore/fold_0/test/inactive/structure_34755.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_zscore/fold_0/test/inactive/structure_43287.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_zscore/fold_0/test/inactive/structure_41484.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_zscore/fold_0/test/inactive/structure_17403.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_zscore/fold_0/test/inactive/structure_39067.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_zscore/fold_0/test/inactive/structure_10556.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_zscore/fold_0/test/inactive/structure_35701.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_zscore/fold_0/test/inactive/structure_50199.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_zscore/fold_0/test/inactive/structure_9885.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_zscore/fold_0/test/inactive/structure_11489.tsv',
  0),
 ('../data/Mateo_2019/C